# GE入图编译与运行

## 前置要求

学习本节前，建议你已经了解以下内容：

- 了解 CANN 软件栈中 Host、Device、AI Core、ACL 等基础概念，能够完成 CANN 环境初始化，并使用 `cmake`、`g++` 等基础命令；
- 了解 Ascend C 算子的基本开发流程，包括 kernel、输入输出张量以及 shape、dtype、format 等张量描述信息；
- 了解主流 AI 框架 PyTorch；
- 了解 C++ 和 Python 基本语法；

## 环境依赖

本节代码支持如下产品型号：

- Ascend 950PR / Ascend 950DT；
- Atlas A3 训练系列产品 / Atlas A3 推理系列产品；
- Atlas A2 训练系列产品 / Atlas A2 推理系列产品。

本节样例默认使用 Ascend 950PR / Ascend 950DT 对应的 `dav-3510` 架构；Atlas A2/A3 环境可使用 `dav-2201`。不同产品环境需保持 CANN、驱动、固件版本配套；请以对应版本官方文档和实际运行结果为准。

运行本节第 3 章 C++ 在线执行样例，需要当前环境已准备好以下软件：

- 已安装与硬件和驱动配套的 CANN 开发环境，并完成环境变量初始化；
- 已安装与 CANN 版本配套的昇腾算子包，样例使用 CANN 内置 `Data`、`Add` 算子需要；
- 系统已安装 `cmake`、`g++` 等基础编译工具。
- 若运行第 3.3 节 TorchAir 扩展样例，则需要额外完成昇腾 PyTorch 适配，并保证 `torch`、`torch_npu`、`torchair` 与当前 CANN、驱动、固件、Python 版本配套。

## 学习目标

完成本节学习后，你应该能够：

- 什么是 GE 入图，哪些算子可以入图。
- 算子入图后，GE 编译阶段做了什么。
- 在线执行链路与离线 OM 执行链路分别如何完成 GE 图运行。
- 入图执行和 eager/单算子执行的调度、编译、适用场景。
- 能基于 Ascend C 自定义算子完成一个最小 GE 入图、编译、运行样例。
- 了解同一个自定义算子如何通过 TorchAir 前端从 PyTorch 图模式接入 GE 图。


## 1. GE 图引擎与算子入图

图引擎（Graph Engine，简称 GE）是昇腾平台计算图编译和运行的控制中心，提供图构建、图编译优化及图执行控制等功能。借助 GE 图引擎能力，PyTorch、TensorFlow、MindSpore、PaddlePaddle 等主流 AI 框架的算法模型可以统一转换为使用 Ascend IR（Ascend Intermediate Representation）表示的计算图（Ascend Graph），并通过 GE 的图编译加速技术提升计算图在昇腾硬件上的执行效率。此外，GE 还提供统一的图开发接口，支持自定义图结构，帮助用户基于昇腾硬件快速部署神经网络业务。

系统了解 GE 的基本概念、原理介绍，以及如何使用 GE 图引擎接口完成图构建、编译和运行，可以参考 [图模式开发指南](https://hiascend.com/document/redirect/CannCommunityGraphguide)。


### 1.1 GE 图里有什么

GE 图可以简单理解为由节点和边组成的计算图。

- **节点**表示一次计算，通常对应一个算子，例如 `Add`、`MatMul`、`AddCustom`；
- **边**表示节点之间的依赖关系，最常见的是数据依赖，也可以有控制依赖；
- **张量描述**记录输入输出的 shape、dtype、format 等信息；
- **属性**记录算子固定参数，例如轴、转置标记等。

后续我们说“算子入图”，就是让一个算子成为 GE 图中的节点，并带上 GE 编译所需的输入输出、张量描述和属性信息。


### 1.2 算子的两种调用方式

从调用入口看，算子通常可以通过单算子调用和 GE 图调用两种方式进入执行链路。

- **单算子调用**：用户程序直接调用某个算子的 API。Host 每调用一次，运行时就围绕这一次算子调用准备输入输出、完成调度并触发执行。它的链路短、反馈快，适合验证单个 kernel 的功能、性能和边界条件。
- **GE 图调用**：先把一组算子描述成计算图，再交给 GE 编译和执行。GE 看到的不只是当前算子，还包括节点之间的数据依赖、shape、dtype、format、输入输出生命周期和整图执行顺序。

因此，算子入图的关键在于算子是否已经作为图节点被 GE 获取，并随整图或子图一起参与编译优化、执行计划生成和运行时调度。

| 对比维度 | 单算子调用 | GE 图调用 |
| --- | --- | --- |
| 调用入口 | Host 直接调用算子 API | Host 触发编译后的计算图或模型执行 |
| 调度粒度 | 每次调用围绕一个算子完成调度 | 以图或子图为粒度统一调度 |
| 编译视角 | 只关注当前算子的输入、输出和属性 | 可以看到上下游依赖和整图结构 |
| 优化范围 | 主要优化单个 kernel 本身 | 可以做图级优化、融合、内存复用、多流并行和模型下沉 |
| 资源申请 | 用户自行申请和管理单次算子调用所需资源 | 图引擎基于整图统一申请和规划资源，空间可在图内不同算子间复用 |
| 适用场景 | 单算子开发、调试、功能验证和局部性能分析 | 整图编译、模型部署、端到端性能优化 |


### 1.3 GE 图编译加速技术

GE 图编译加速技术的核心，是把多次即时执行的算子调用转换为一张可整体分析、整体编译、整体下发的计算图。相比 Host 逐个下发算子的执行方式，图模式具备全局视角，GE 可以结合节点依赖关系、张量生命周期、执行顺序和硬件资源做整图优化。

常见的性能收益主要来自以下手段：

- **计算图优化**：在编译期简化图结构，删除无效节点，折叠常量表达式，消除重复计算，并为后续算子选择、执行计划生成提供更稳定的图结构。
- **算子融合与小 shape 算子优化**：把适合合并的相邻计算融合到更少的执行任务中，减少中间 Tensor 读写和 kernel 启动次数；对于小 shape、轻量计算等调度开销占比较高的场景，尽量降低 Host 侧反复下发带来的额外开销。
- **多流并行**：根据图中的数据依赖关系识别可并行分支，把互不依赖的任务安排到不同 stream 上执行，提高设备侧并行度和硬件利用率。
- **内存复用**：基于 Tensor 生命周期统一规划中间 Tensor 和 workspace，让生命周期不重叠的内存块复用，降低整图峰值内存占用。
- **模型下沉**：将编译后的整图或子图下沉到设备侧执行，运行时由 Host 一次触发模型执行，减少 Host 与 Device 之间的调度交互。

因此，GE 的性能收益来自整图编译后对调度、并行、内存和执行计划的统一优化。对于小算子较多、图结构稳定、存在上下游优化空间或中间 Tensor 较多的场景，图编译加速技术更容易体现收益。


## 2. GE 图如何编译和运行

GE 图的完整使用流程可以概括为：先在用户侧创建 Graph，描述输入、算子节点和输出之间的计算关系；再由 GE 对图进行编译优化，生成可调度的执行计划；最后根据使用场景选择在线执行或离线 OM 执行链路。


### 2.1 创建 GE 计算图

构图阶段的目标，是把用户希望执行的计算关系表达成 GE 的 `Graph` 对象。图中需要明确输入、输出、算子节点、数据依赖、控制依赖，以及每个张量的 shape、dtype、format 等描述信息。

以 C++ 构图接口为例，典型过程包括：

1. 创建 `ge::Graph` 对象；
2. 创建 `ge::op::Data` 等输入节点，并设置输入输出 Tensor 描述；
3. 创建 CANN 内置算子节点或自定义算子节点，例如 `ge::op::Add`、`ge::op::AddCustom`；
4. 通过 `set_input_*` 接口连接节点输入，形成数据依赖；
5. 使用 `SetInputs`、`SetOutputs` 指定整张图对外暴露的输入和输出。

无论是在线执行还是离线编译，算子都需要先成为 GE 图中的节点。对于自定义算子，构图时写入图中的 op type、输入输出个数、属性名称和张量描述，必须与算子原型定义保持一致。只有 GE 能识别这个节点，后续才有机会对它做 shape 推导、实现选择、任务生成或运行时调度。


### 2.2 GE 编译流程概述

GE 编译器以 AscendIR / Graph 为输入，把图转换为可执行的模型或执行计划。核心编译流程包括预处理、图优化、引擎分区和构建四个阶段。

<p align="center"><img src="./images/03_ge_compile_launch/2_2_ge_compile_flow.svg" alt="GE 编译流程概述" width="60%"></p>

1. **预处理**：对图结构做规范化处理，执行 shape / dtype / format 推导，补齐后续优化和内存规划需要的张量描述。
2. **图优化**：在整图视角下执行常量折叠、公共子表达式消除、死代码消除、算子融合、精度和格式调整等优化。
3. **引擎分区**：根据节点能力和执行后端把图划分到不同引擎或子图中，再对子图做引擎相关优化。
4. **构建执行计划**：完成流分配、内存规划、任务生成等工作，形成 GE 执行器可以加载和调度的编译结果。离线场景还会把编译结果序列化为 OM。

在线执行会在当前进程内完成图编译、加载和运行；离线编译会把编译结果保存为 OM 或模型缓冲区，部署阶段再加载执行。


### 2.3 GE 图的两种执行链路

GE 图完成构建和编译后，可以在当前进程内直接在线执行，也可以先离线编译成 OM，再由部署程序通过 ACL 加载执行。两种链路共用 GE 的图理解、图优化和任务生成能力，差异主要体现在编译产物保存位置、部署入口和运行时接口。

#### 2.3.1 在线执行：`Session::RunGraph`

在线执行指程序在运行时构造 `ge::Graph`，并在当前进程内触发 GE 完成图准备和执行。后文第 3 章给出的 GE C++ 直接构图样例，以及 TorchAir 前端入图样例，都属于这条链路。GE C++ 直接构图样例的核心调用是：

```cpp
ge::Session session(options);
session.AddGraph(GRAPH_ID, graph);
session.RunGraph(GRAPH_ID, inputs, outputs);
```

在线执行不需要显式保存 OM 文件，但并不表示没有图编译或图优化过程。GE 仍会围绕整张图完成一系列准备工作：

- 根据 op type 查找算子定义，并检查节点输入、输出和属性是否合法；
- 执行 shape、dtype、format 等推导，补齐后续执行需要的张量描述；
- 进行图优化、算子融合、常量折叠、冗余节点删除等图级处理；
- 为节点选择可用实现，准备执行任务、stream、event、workspace 等运行信息；
- 在 `RunGraph` 执行时按图依赖调度各个节点。


#### 2.3.2 离线编译：生成 OM 并用 ACL 执行

离线编译链路会把 `Graph` 或框架转换得到的中间表示编译成适配昇腾 AI 处理器的离线模型。GE Python 的 `offline_compile` 模块提供 `build_initialize`、`build_model`、`save_model`、`build_finalize` 等接口；C/C++ 接口中，`aclgrphBuildModel` 可以将输入 Graph 编译为离线模型缓冲区；也可以使用 `atc` 从模型或中间表示生成 `.om` 文件。

OM 是离线编译链路的产物，通常包含编译后的计算图、任务信息、权重和运行所需的资源描述。生成 OM 之后，运行程序不再调用 `Session::RunGraph` 构图执行，而是通过 ACL 加载并执行模型：

1. 初始化 ACL，并设置 Device / Context；
2. 调用 `aclmdlLoadFromFile` 或 `aclmdlLoadFromMem` 加载 OM，获取 `modelId`；
3. 根据模型输入输出描述申请 Device 内存；
4. 创建输入、输出 `aclmdlDataset`，并把输入数据拷贝到 Device；
5. 调用 `aclmdlExecute` 同步执行，或调用异步执行接口后进行 stream 同步；
6. 将输出从 Device 拷回 Host，并释放 Dataset、模型和设备资源。

离线 OM 场景下，CANN 内置算子会作为 GE 编译后的执行任务进入 OM。常规离线自定义算子也需要以 GE 可编译的方式提供原型、推导、kernel 和算子包安装信息，由 GE 编译进 OM。

采用离线编译链路，调用方式和交付件会从“运行时构图 + 在线执行程序”变成“编译期生成 OM + 部署程序加载 OM”，常见差异如下：

| 维度 | 在线执行 | 离线 OM 执行 |
| --- | --- | --- |
| 图来源 | Host 代码或 TorchAir 在运行时构建/捕获图 | 构建或发布阶段由 Graph、框架模型或中间表示编译出 OM |
| 主要交付件 | Host 可执行程序或 Python 前端脚本、自定义算子动态库/算子包、运行环境配置 | `.om` 模型文件、ACL 模型加载与执行程序、模型输入输出描述、必要的自定义算子编译/安装产物 |
| 运行入口 | `Session::RunGraph` 或 TorchAir 后端在线触发 GE 执行 | `aclmdlLoadFromFile` / `aclmdlExecute` 等 ACL 模型接口 |
| 编译时机 | 运行进程内完成图准备、优化和执行计划生成 | 部署前生成 OM，运行时主要负责加载和执行 OM |
| 自定义算子要求 | 运行时能被 GE 发现原型、推导和 Execute 执行实现 | 编译 OM 时能被 GE 发现原型、推导和 kernel/算子包；部署侧还需携带或安装 OM 执行所依赖的自定义算子产物 |


## 3. 自定义算子接入 GE 图


---
### 3.1 环境准备

正式开始学习之前，先对 Jupyter 环境进行初始化。以下代码会重新创建本节使用的 `Sources` 目录，并将 CANN 环境变量导入当前 Notebook 进程，便于后续逐段写入代码、编译和运行样例。


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# 重建 Sources 目录，避免旧的源文件、CMake 配置或构建产物影响本次样例。
source_dir = Path("Sources")
if source_dir.exists():
    shutil.rmtree(source_dir)
source_dir.mkdir(exist_ok=True)

# 按常见安装路径查找 CANN 环境初始化脚本。
set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/ascend-toolkit/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

if not Path(set_env).exists():
    raise FileNotFoundError(f"未找到 CANN 环境初始化脚本: {set_env}")

# 将 set_env.sh 中导出的环境变量同步到当前 Notebook 进程，后续 cmake 和 demo 会直接使用。
result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")
print("set_env:", set_env)
print("Sources:", source_dir.resolve())


---
### 3.2 算子入图实例：CANN 内置算子 Add + 自定义算子 AddCustom 混合样例

本节通过一个 CANN 内置 ` Add 算子` 节点和一个 `自定义算子 AddCustom` 节点展示两类算子如何进入同一张 GE 图：CANN 内置算子直接使用 CANN 已提供的算子原型和实现，自定义算子则需要先声明原型，再提供 shape/dtype 推导、执行入口、设备侧 kernel 和注册映射。

这个样例中的 `AddCustom` 采用 **Execute 回调型自定义算子入图**：GE 图中出现 `AddCustom` 节点后，通过注册映射找到用户实现的执行类；图运行到该节点时进入 `EagerExecuteOp::Execute()`，由用户代码从 GE 执行上下文中读取输入、创建输出 Tensor，并启动 Ascend C kernel 完成计算。


本节先从 GE C++ 入口直接构图, 相关样例可参考 asc-devkit 的 [GE 自定义算子入图样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/02_features/00_framework/03_ge)。

两类算子入图流程对比如下图：

<p align="center"><img src="./images/03_ge_compile_launch/3_2_builtin_custom_flow.svg" alt="CANN 内置算子与自定义算子入图流程对比" width="60%"></p>

实例中先构造一个 CANN 内置 `Add` 节点，再构造一个自定义 `AddCustom` 节点：

<p align="center"><img src="./images/03_ge_compile_launch/3_2_builtin_custom_graph.svg" alt="CANN 内置 Add 与 AddCustom 混合计算图" width="60%"></p>

整张图的最终输出是：

$$
\large output = (x + y) + y
$$


---
#### 3.2.1 声明 GE 算子原型

CANN 内置 `Add` 的原型、推导规则和执行实现已经由 CANN 内置算子包提供，样例中只需要包含 CANN 内置 `Data` 和 `Add` 对应头文件，并创建 `ge::op::Add` 节点。`AddCustom` 是用户自定义算子，需要使用 `REG_OP` 声明 GE 能识别的算子原型，包括算子类型、输入、输出以及支持的数据类型。后续构图时创建的 `ge::op::AddCustom` 节点，其 op type、输入输出名称和张量类型需要与这里的 `REG_OP(AddCustom)` 声明保持一致。


In [ ]:
%%writefile Sources/add_custom_proto.h
#ifndef ADD_CUSTOM_PROTO_H_
#define ADD_CUSTOM_PROTO_H_

#include "graph/operator_reg.h"

namespace ge {
// 声明 AddCustom 算子原型：
// - 算子名：AddCustom
// - 输入：x、y
// - 输出：z
// - 数据类型：x、y、z 均为 DT_FLOAT
REG_OP(AddCustom)
    .INPUT(x, TensorType({DT_FLOAT}))
    .INPUT(y, TensorType({DT_FLOAT}))
    .OUTPUT(z, TensorType({DT_FLOAT}))
    .OP_END_FACTORY_REG(AddCustom);
}  // namespace ge

#endif  // ADD_CUSTOM_PROTO_H_


---
#### 3.2.2 实现 AddCustom 自定义算子

自定义算子接入 GE 图执行时，需要提供设备侧计算实现、输出信息推导逻辑、执行入口以及节点到执行类的映射关系。本样例的自定义算子实现由 `add_custom.asc` 承载。

`add_custom.asc` 包含以下内容：

1. Ascend C 核函数：运行在 NPU 上，执行 `z = x + y`；
2. GE 执行类：实现 `Execute()`，在 GE 运行到 `AddCustom` 节点时读取输入、创建输出 Tensor 并启动核函数；
3. shape / dtype 推导逻辑：根据输入 Tensor 描述推导输出 Tensor 的 shape 和 dtype；
4. 映射注册语句：通过 `REG_AUTO_MAPPING_OP(AddCustom)` 将图中的 `AddCustom` 节点关联到用户实现的执行类。

下面从头文件和常量定义开始说明。


In [ ]:
%%writefile Sources/add_custom.asc
#include <iostream>

#include "acl/acl_rt.h"
#include "graph/custom_op.h"
#include "kernel_operator.h"

namespace {
// 本样例固定处理 8 * 2048 个 float 元素。
// BLOCK_NUM 表示启动的 block 数，BLOCK_LENGTH 表示每个 block 处理的元素数。
constexpr uint32_t BLOCK_NUM = 8U;
constexpr uint32_t BLOCK_LENGTH = 2048U;
constexpr uint32_t EXPECTED_ELEMENT_COUNT = BLOCK_NUM * BLOCK_LENGTH;

// AddCustom 的输入输出索引：x、y -> z。
// 后续 Execute、InferShape、InferDataType 都通过这些索引访问对应 Tensor。
constexpr size_t INPUT_X_INDEX = 0U;
constexpr size_t INPUT_Y_INDEX = 1U;
constexpr size_t OUTPUT_Z_INDEX = 0U;
}  // namespace


##### 3.2.2.1 写入 Ascend C 核函数

这个核函数固定启动 `8` 个 block，每个 block 处理 `2048` 个 `float`。

每个 block 内部需要做如下处理：

1. 根据 `block_idx` 找到自己负责的数据段；
2. 从 GM 搬入 `x`、`y` 到 UB；
3. 执行 `AscendC::Add`，再把结果写回 GM。


In [ ]:
%%writefile -a Sources/add_custom.asc

template <uint32_t blockLength>
__global__ __vector__ void add_custom(__gm__ float *x, __gm__ float *y, __gm__ float *z)
{
    AscendC::InitSocState();

    // 每个 block 只处理自己负责的连续数据段。
    AscendC::GlobalTensor<float> xGm, yGm, zGm;
    xGm.SetGlobalBuffer(x + block_idx * blockLength, blockLength);
    yGm.SetGlobalBuffer(y + block_idx * blockLength, blockLength);
    zGm.SetGlobalBuffer(z + block_idx * blockLength, blockLength);

    // 在 UB 上分配临时 Tensor，完成 GM -> UB -> GM 的典型向量计算流程。
    AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    AscendC::LocalTensor<float> xLocal = ubAllocator.Alloc<float, blockLength>();
    AscendC::LocalTensor<float> yLocal = ubAllocator.Alloc<float, blockLength>();
    AscendC::LocalTensor<float> zLocal = ubAllocator.Alloc<float, blockLength>();

    // 搬入输入数据。
    AscendC::DataCopy(xLocal, xGm, blockLength);
    AscendC::DataCopy(yLocal, yGm, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();

    // 执行逐元素加法 z = x + y。
    AscendC::Add(zLocal, xLocal, yLocal, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();

    // 将计算结果写回 GM。
    AscendC::DataCopy(zGm, zLocal, blockLength);
    AscendC::PipeBarrier<PIPE_ALL>();
}


##### 3.2.2.2 写入 GE 执行类框架

本样例的 `AddCustom` 执行类同时继承两个基类：

- `EagerExecuteOp`：提供 GE 执行阶段入口 `Execute()`；
- `ShapeInferOp`：提供图编译或执行准备阶段需要的 shape / dtype 推导接口。

GE 通过后续的 `REG_AUTO_MAPPING_OP(AddCustom)` 把图中的 `AddCustom` 节点映射到这个执行类。先写入类声明和 `Execute()` 的输入校验部分。


In [ ]:
%%writefile -a Sources/add_custom.asc

namespace ge {
// EagerExecuteOp 提供运行时 Execute 入口，ShapeInferOp 提供输出信息推导入口。
class AddCustom : public EagerExecuteOp, public ShapeInferOp {
public:
    graphStatus Execute(gert::EagerOpExecutionContext *ctx) override
    {
        if (ctx == nullptr) {
            std::cerr << "Execute context is null." << std::endl;
            return GRAPH_FAILED;
        }

        // 从 GE 执行上下文中读取 AddCustom 的两个输入 Tensor。
        const gert::Tensor *x = ctx->GetInputTensor(INPUT_X_INDEX);
        const gert::Tensor *y = ctx->GetInputTensor(INPUT_Y_INDEX);
        if ((x == nullptr) || (y == nullptr)) {
            std::cerr << "Input tensor is null." << std::endl;
            return GRAPH_FAILED;
        }

        // 本教学样例只覆盖 float32，和 add_custom_proto.h 中的 DT_FLOAT 保持一致。
        if ((x->GetDataType() != DT_FLOAT) || (y->GetDataType() != DT_FLOAT)) {
            std::cerr << "AddCustom only supports float in this sample." << std::endl;
            return GRAPH_FAILED;
        }

        // kernel 固定按 8 * 2048 个元素组织计算，运行前检查输入规模。
        if ((x->GetShapeSize() != EXPECTED_ELEMENT_COUNT) || (y->GetShapeSize() != EXPECTED_ELEMENT_COUNT)) {
            std::cerr << "AddCustom expects both inputs to have 8 * 2048 elements." << std::endl;
            return GRAPH_FAILED;
        }


##### 3.2.2.3 执行 AddCustom 计算

`Execute()` 是 `AddCustom` 真正执行计算的位置。GE 运行到 `AddCustom` 节点后，会调用这里的用户代码；用户代码从执行上下文中读取输入 Tensor，创建输出 Tensor，并将输入、输出的数据地址传给 Ascend C 核函数。


In [ ]:
%%writefile -a Sources/add_custom.asc

        // 输出 z 与输入 x 保持相同 shape、format 和 dtype；GE 为 z 创建可写 Tensor。
        gert::Tensor *z = ctx->MallocOutputTensor(OUTPUT_Z_INDEX, x->GetShape(), x->GetFormat(), x->GetDataType());
        if (z == nullptr) {
            std::cerr << "Failed to malloc output tensor." << std::endl;
            return GRAPH_FAILED;
        }

        // 使用 GE 当前执行流启动 Ascend C kernel，使 AddCustom 计算纳入 GE 调度。
        add_custom<BLOCK_LENGTH><<<BLOCK_NUM, nullptr, ctx->GetStream()>>>(
            (float *)const_cast<void *>(x->GetAddr()),
            (float *)const_cast<void *>(y->GetAddr()),
            (float *)z->GetAddr());
        return GRAPH_SUCCESS;
    }


##### 3.2.2.4 写入 shape / dtype 推导和注册语句

GE 处理 `AddCustom` 节点时需要拿到输出 Tensor 的描述信息，后续才能完成图优化、内存规划和执行准备。本样例让 `AddCustom` 类继承 `ShapeInferOp`，并在类中实现 `InferShape()` 和 `InferDataType()`：

- `InferShape()`：把输出 `z` 的 shape 设置为输入 `x` 的 shape；
- `InferDataType()`：把输出 `z` 的 dtype 设置为输入 `x` 的 dtype。

逐元素加法的输出 shape 和 dtype 都与输入 `x` 保持一致，因此这里的推导逻辑比较直接。最后通过 `REG_AUTO_MAPPING_OP(AddCustom)` 完成映射注册：GE 图中遇到 `AddCustom` 节点时，就能找到这个同时提供推导和执行入口的 `AddCustom` 类。


In [ ]:
%%writefile -a Sources/add_custom.asc

    graphStatus InferShape(gert::InferShapeContext *ctx) override
    {
        if (ctx == nullptr) {
            return GRAPH_FAILED;
        }
        const gert::Shape *xShape = ctx->GetInputShape(INPUT_X_INDEX);
        gert::Shape *zShape = ctx->GetOutputShape(OUTPUT_Z_INDEX);
        if ((xShape == nullptr) || (zShape == nullptr)) {
            return GRAPH_FAILED;
        }

        // 逐元素加法不改变张量形状，输出 z 的 shape 跟随输入 x。
        *zShape = *xShape;
        return GRAPH_SUCCESS;
    }

    graphStatus InferDataType(gert::InferDataTypeContext *ctx) override
    {
        if (ctx == nullptr) {
            return GRAPH_FAILED;
        }

        // 输出 z 的 dtype 跟随输入 x，与原型中的 DT_FLOAT 约束保持一致。
        return ctx->SetOutputDataType(OUTPUT_Z_INDEX, ctx->GetInputDataType(INPUT_X_INDEX));
    }
};

// 将 GE 图中的 AddCustom 节点映射到上面的 AddCustom 执行类。
REG_AUTO_MAPPING_OP(AddCustom);
}  // namespace ge


---
#### 3.2.3 构造并运行 GE 图

Host 侧程序负责完成 GE 初始化、图构造、输入准备、图执行和结果校验。主要步骤如下：

1. 初始化 GE；
2. 构造包含一个 CANN 内置 `Add` 节点和一个自定义 `AddCustom` 节点的 GE 图；
3. 准备输入 Tensor；
4. 调用 `Session::RunGraph`；
5. 校验输出是否等于 golden。

下面从头文件和常量定义开始说明。


In [ ]:
%%writefile Sources/graph.cpp
#include <algorithm>
#include <cmath>
#include <cstdint>
#include <cstdlib>
#include <iostream>
#include <iterator>
#include <map>
#include <memory>
#include <vector>

#include "add_custom_proto.h"
#include "ge/ge_api.h"
#include "graph.h"
#include "array_ops.h"
#include "elewise_calculation_ops.h"
#include "tensor.h"
#include "types.h"

using ge::Operator;

namespace {
constexpr uint32_t GRAPH_ID = 0U;

// 与 AddCustom kernel 保持一致：整图输入输出均为 [8, 2048] 的 float32 Tensor。
constexpr int64_t BLOCK_NUM = 8;
constexpr int64_t BLOCK_LENGTH = 2048;
constexpr size_t ELEMENT_COUNT = static_cast<size_t>(BLOCK_NUM * BLOCK_LENGTH);

// x、y 的初始化比例；最终输出为 x + y + y。
constexpr float X_SCALE = 0.1F;
constexpr float Y_SCALE = 0.2F;
constexpr float EPSILON = 1e-6F;


##### 3.2.3.1 构造 GE 图

这里创建两个输入节点 `x`、`y`，然后依次放入一个 CANN 内置 `Add` 节点和一个自定义 `AddCustom` 节点：

- `add_builtin = Add(x, y)`：CANN 内置算子，构图时直接创建 `ge::op::Add` 并连接输入；
- `add_custom = AddCustom(add_builtin, y)`：用户自定义算子，构图写法和普通节点一致，但它依赖前面生成的 `AddCustom` 原型、推导逻辑、执行入口和注册映射。

最终图输出是 `add_custom`。这段代码可以直接看到两类算子入图时的差异：CANN 内置算子的定义和实现来自 CANN，自定义算子的定义和实现来自本节生成的用户代码。


In [ ]:
%%writefile -a Sources/graph.cpp

std::unique_ptr<ge::Graph> BuildGraph()
{
    // 图中所有节点使用相同 Tensor 描述：[8, 2048]、ND format、float32。
    ge::TensorDesc tensorDesc(ge::Shape({BLOCK_NUM, BLOCK_LENGTH}), ge::FORMAT_ND, ge::DT_FLOAT);

    // Data 节点表示整张图的外部输入 x。
    auto x = ge::op::Data("x");
    x.update_input_desc_x(tensorDesc);
    x.update_output_desc_y(tensorDesc);

    // Data 节点表示整张图的外部输入 y。
    auto y = ge::op::Data("y");
    y.update_input_desc_x(tensorDesc);
    y.update_output_desc_y(tensorDesc);

    // CANN 内置 Add 入图：直接创建 CANN 已支持的 ge::op::Add 节点。
    auto addBuiltin = ge::op::Add("add_builtin").set_input_x1(x).set_input_x2(y);
    addBuiltin.update_output_desc_y(tensorDesc);

    // 自定义 AddCustom 入图：节点名称和输入输出需要与 add_custom_proto.h 中的原型一致。
    auto addCustom = ge::op::AddCustom("add_custom").set_input_x(addBuiltin).set_input_y(y);
    addCustom.update_output_desc_z(tensorDesc);

    // 声明整张图的外部输入和最终输出。
    std::vector<Operator> inputs{x, y};
    std::vector<Operator> outputs{addCustom};
    auto graph = std::make_unique<ge::Graph>("BuiltinAndCustomAddGraph");
    graph->SetInputs(inputs).SetOutputs(outputs);
    return graph;
}


##### 3.2.3.2 构造输入和 golden

`Session::RunGraph` 需要两路输入 Tensor。样例同时按图结构 `output = AddCustom(Add(x, y), y)` 计算期望输出 `golden`，后续用它校验 GE 图执行结果。


In [ ]:
%%writefile -a Sources/graph.cpp

bool BuildInputTensor(float scale, ge::Tensor &tensor)
{
    ge::TensorDesc tensorDesc(ge::Shape({BLOCK_NUM, BLOCK_LENGTH}), ge::FORMAT_ND, ge::DT_FLOAT);
    tensor = ge::Tensor(tensorDesc);

    // 构造 Host 侧输入数据，SetData 会把数据写入 ge::Tensor。
    std::vector<float> data(ELEMENT_COUNT);
    for (size_t i = 0U; i < data.size(); ++i) {
        data[i] = static_cast<float>(i) * scale;
    }
    const auto setDataRet = tensor.SetData(reinterpret_cast<const uint8_t *>(data.data()),
        data.size() * sizeof(float));
    if (setDataRet != ge::GRAPH_SUCCESS) {
        std::cerr << "SetData failed, ret: " << setDataRet << std::endl;
        return false;
    }
    return true;
}

std::vector<float> BuildGolden()
{
    // 图结构为 output = AddCustom(Add(x, y), y)，因此 golden = x + y + y。
    std::vector<float> golden(ELEMENT_COUNT);
    for (size_t i = 0U; i < golden.size(); ++i) {
        golden[i] = static_cast<float>(i) * X_SCALE + static_cast<float>(i) * Y_SCALE +
            static_cast<float>(i) * Y_SCALE;
    }
    return golden;
}


##### 3.2.3.3 校验输出

`CheckOutputTensor` 会先检查输出 shape 和数据长度，再使用 epsilon 容差逐元素比较输出值和 golden。


In [ ]:
%%writefile -a Sources/graph.cpp

bool CheckOutputTensor(const ge::Tensor &outputTensor)
{
    // 先检查输出 Tensor 描述，避免后续按错误 shape 解释数据。
    const auto tensorDesc = outputTensor.GetTensorDesc();
    const auto dims = tensorDesc.GetShape().GetDims();
    if ((dims.size() != 2U) || (dims[0] != BLOCK_NUM) || (dims[1] != BLOCK_LENGTH)) {
        std::cerr << "Unexpected output shape." << std::endl;
        return false;
    }
    if (outputTensor.GetSize() < ELEMENT_COUNT * sizeof(float)) {
        std::cerr << "Unexpected output data size: " << outputTensor.GetSize() << std::endl;
        return false;
    }

    const auto *outputData = reinterpret_cast<const float *>(outputTensor.GetData());
    std::vector<float> output(outputData, outputData + ELEMENT_COUNT);
    std::vector<float> golden = BuildGolden();

    // 只打印前 20 个元素，便于观察输出模式。
    auto printTensor = [](const std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20U;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");

    // 使用容差逐元素校验，避免不同执行路径的 float32 舍入差异导致误报。
    for (size_t i = 0U; i < ELEMENT_COUNT; ++i) {
        const float diff = std::abs(output[i] - golden[i]);
        if (diff > EPSILON) {
            std::cerr << "Unexpected output at index " << i << ", expected " << golden[i] << ", got "
                      << output[i] << ", diff " << diff << std::endl;
            return false;
        }
    }
    return true;
}
}  // namespace


##### 3.2.3.4 主函数：初始化 GE、运行图、释放资源

`Session::RunGraph` 是真正触发整张图执行的位置。

设备号默认取环境变量 `ASCEND_DEVICE_ID`；如果没有设置，则使用 `1`，与原样例保持一致。


In [ ]:
%%writefile -a Sources/graph.cpp

int main(int argc, char *argv[])
{
    (void)argc;
    (void)argv;

    // 默认使用 ASCEND_DEVICE_ID 指定的设备；未设置时沿用样例默认设备 1。
    const char *deviceId = std::getenv("ASCEND_DEVICE_ID");
    if (deviceId == nullptr) {
        deviceId = "1";
    }
    std::map<ge::AscendString, ge::AscendString> options = {
        {"ge.exec.deviceId", deviceId},
        {"ge.graphRunMode", "1"},
    };

    // 初始化 GE，后续 Session 构图和 RunGraph 都依赖该初始化过程。
    const auto initRet = ge::GEInitialize(options);
    if (initRet != ge::SUCCESS) {
        std::cerr << "GEInitialize failed, ret: " << initRet << std::endl;
        return 1;
    }

    int retCode = 0;
    ge::Session session(options);
    auto graph = BuildGraph();
    if (graph == nullptr) {
        std::cerr << "BuildGraph failed." << std::endl;
        (void)ge::GEFinalize();
        return 1;
    }

    // 将内存中的 Graph 加入 Session，GE 会在后续 RunGraph 时完成图准备和执行。
    const auto addGraphRet = session.AddGraph(GRAPH_ID, *graph);
    if (addGraphRet != ge::SUCCESS) {
        std::cerr << "AddGraph failed, ret: " << addGraphRet << std::endl;
        retCode = 1;
    } else {
        ge::Tensor xTensor;
        ge::Tensor yTensor;
        if (!BuildInputTensor(X_SCALE, xTensor) || !BuildInputTensor(Y_SCALE, yTensor)) {
            std::cerr << "Build input tensors failed." << std::endl;
            retCode = 1;
        } else {
            std::vector<ge::Tensor> inputs{xTensor, yTensor};
            std::vector<ge::Tensor> outputs;

            // 在线执行整张 GE 图，输出写入 outputs。
            const auto runRet = session.RunGraph(GRAPH_ID, inputs, outputs);
            if (runRet != ge::SUCCESS) {
                std::cerr << "RunGraph failed, ret: " << runRet << std::endl;
                retCode = 1;
            } else if (outputs.empty()) {
                std::cerr << "RunGraph succeeded but outputs is empty." << std::endl;
                retCode = 1;
            } else if (!CheckOutputTensor(outputs[0])) {
                retCode = 1;
            } else {
                std::cout << "GE builtin/custom add sample success." << std::endl;
            }
        }
        (void)session.RemoveGraph(GRAPH_ID);
    }

    const auto finalizeRet = ge::GEFinalize();
    if (finalizeRet != ge::SUCCESS) {
        std::cerr << "GEFinalize failed, ret: " << finalizeRet << std::endl;
        return 1;
    }
    return retCode;
}


---
#### 3.2.4 写入 CMake 编译配置

编译会生成两类产物：

- `libadd_custom_op.so`：自定义算子动态库，供 GE 加载；
- `demo`：Host 侧构图和运行程序。

注意：`configure_file` 会把 `add_custom_proto.h` 复制到 `output/op_graph/include`，动态库会输出到 `output/op_graph/lib/linux/<arch>`。`demo` 还会包含 CANN 内置算子的 op_graph 头文件，用于识别 CANN 内置 `Data` 和 `Add`。


In [ ]:
%%writefile Sources/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

# NPU 架构参数需要与目标硬件匹配；样例默认使用 dav-3510，Atlas A2/A3 可使用 dav-2201。
set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510, dav-2201")

# ASC 语言支持由 CANN 提供，用于编译 .asc 自定义算子源文件。
find_package(ASC REQUIRED)
project(builtin_custom_add_notebook LANGUAGES ASC CXX)

# 自定义算子动态库和原型头文件按 GE 运行时可发现的目录结构输出。
set(CUSTOM_OP_OUTPUT_DIR "${CMAKE_BINARY_DIR}/output/op_graph/lib/linux/${CMAKE_SYSTEM_PROCESSOR}")
set(CUSTOM_OP_INCLUDE_DIR "${CMAKE_BINARY_DIR}/output/op_graph/include")
file(MAKE_DIRECTORY "${CUSTOM_OP_OUTPUT_DIR}" "${CUSTOM_OP_INCLUDE_DIR}")

# graph.cpp 需要包含 AddCustom 原型，因此把原型头文件复制到 output/op_graph/include。
set(OP_PROTO_HEADER_SOURCE_FILE "${CMAKE_SOURCE_DIR}/add_custom_proto.h")
configure_file("${OP_PROTO_HEADER_SOURCE_FILE}" "${CUSTOM_OP_INCLUDE_DIR}/add_custom_proto.h" COPYONLY)

# add_custom_op 是 GE 运行时加载的自定义算子动态库。
add_library(add_custom_op SHARED
    add_custom.asc
)
target_compile_options(add_custom_op PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)
target_compile_definitions(add_custom_op PRIVATE _GLIBCXX_USE_CXX11_ABI=0)
target_include_directories(add_custom_op PRIVATE
    "$ENV{ASCEND_HOME_PATH}/include"
)
target_link_directories(add_custom_op PRIVATE "$ENV{ASCEND_HOME_PATH}/lib64")

set_target_properties(add_custom_op PROPERTIES
    LIBRARY_OUTPUT_DIRECTORY "${CUSTOM_OP_OUTPUT_DIR}"
)

# demo 是 Host 侧程序，负责构造 GE 图并调用 Session::RunGraph 在线执行。
add_executable(demo
    graph.cpp
)
target_compile_definitions(demo PRIVATE _GLIBCXX_USE_CXX11_ABI=0)
target_include_directories(demo PRIVATE
    "${CMAKE_SOURCE_DIR}"
    "$ENV{ASCEND_HOME_PATH}/include"
    "$ENV{ASCEND_HOME_PATH}/include/graph"
    "$ENV{ASCEND_HOME_PATH}/opp/built-in/op_graph/inc"
    "$ENV{ASCEND_HOME_PATH}/opp/built-in/op_proto/inc"
)
target_link_directories(demo PRIVATE "$ENV{ASCEND_HOME_PATH}/lib64")
target_link_libraries(demo PRIVATE
    graph
    ge_runner
    graph_base
)


---
#### 3.2.5 编译和运行

执行以下命令编译样例。这里沿用参考模板的写法，使用 `!cmake -S ... -B ...` 配置构建目录，再通过 `!cmake --build ...` 编译。

样例默认使用 Ascend 950PR / Ascend 950DT 对应的 `dav-3510` 架构；Atlas A2/A3 环境可将其改为 `dav-2201`。


In [ ]:
# 配置并编译样例工程；样例默认使用 dav-3510，Atlas A2/A3 环境可改为 dav-2201。
!cmake -S Sources -B Sources/build -DCMAKE_ASC_ARCHITECTURES=dav-3510
!cmake --build Sources/build -j


运行前需要把 `Sources/build/output` 加入 `ASCEND_CUSTOM_OPP_PATH`，这样 GE 才能在运行图时找到 `AddCustom` 的自定义算子动态库。CANN 内置 `Add` 来自 CANN 内置算子包，不需要这一步。


In [ ]:
import os
from pathlib import Path

# 将自定义算子输出目录加入 ASCEND_CUSTOM_OPP_PATH，GE 运行时通过该路径发现 AddCustom 动态库。
custom_opp_path = str((Path("Sources") / "build" / "output").resolve())
old_opp_path = os.environ.get("ASCEND_CUSTOM_OPP_PATH", "")
os.environ["ASCEND_CUSTOM_OPP_PATH"] = custom_opp_path if not old_opp_path else f"{custom_opp_path}:{old_opp_path}"
print("ASCEND_CUSTOM_OPP_PATH=", os.environ["ASCEND_CUSTOM_OPP_PATH"])


再执行以下代码，运行 GE 图样例。

如果成功，会看到类似输出：

```text
Output: 0 0.5 1 1.5 ...
Golden: 0 0.5 1 1.5 ...
GE builtin/custom add sample success.
```


In [ ]:
# 执行 Host 侧 demo，触发 Session::RunGraph 在线运行整张 GE 图。
!Sources/build/demo


---
### 3.3 可选扩展：通过 TorchAir 前端接入 AddCustom

上一节由 C++ Host 代码直接创建 `ge::Graph` 并调用 `Session::RunGraph`。TorchAir 接入方式沿用 GE 侧自定义算子的接口要求和实现思路，把前端入口替换为 PyTorch 图模式：模型中调用 `torch.ops.ascendc_ops.ascendc_add`，`torch.compile` 捕获该调用，TorchAir converter 再把 FX 节点转换为 GE `AddCustom` 节点。这个样例仍属于在线图执行链路，运行时由 TorchAir 后端完成图捕获、编译和执行，不生成离线 OM 文件。

| 阶段 | GE C++ 直接构图 | GE + TorchAir 前端入图 |
| --- | --- | --- |
| 前端入口 | C++ Host 程序创建 `ge::Graph` | PyTorch 模型调用 `torch.ops.*`，由 `torch.compile` 捕获 |
| GE 节点创建方式 | 显式创建 `ge::op::AddCustom` | converter 调用 `torchair.ge.custom_op("AddCustom", ...)` |
| 新增适配内容 | `BuildGraph()`、输入 Tensor 构造、`Session::RunGraph` | PyTorch schema / Meta、TorchAir converter、`torch.compile` 调用 |
| GE 侧要求 | `REG_OP(AddCustom)`、shape / dtype 推导、`Execute()`、Ascend C kernel、`REG_AUTO_MAPPING_OP(AddCustom)` | 同样需要 `REG_OP(AddCustom)`、shape / dtype 推导、`Execute()`、Ascend C kernel、`REG_AUTO_MAPPING_OP(AddCustom)` |

参考 asc-devkit 的 [ge_torchair 样例](https://gitcode.com/cann/asc-devkit/tree/master/examples/01_simd_cpp_api/02_features/00_framework/03_ge)，本节把前面合在一个 `add_custom.asc` 中的逻辑拆成三个文件：Ascend C kernel、PyTorch schema / Meta、GE 原型与执行实现。


#### 3.3.1 TorchAir 扩展样例目录

`TorchAirSources` 与上一节的 `Sources` 相互独立，便于在没有 TorchAir 环境时继续运行主样例。


In [ ]:
import shutil
from pathlib import Path

torchair_source_dir = Path("TorchAirSources")
if torchair_source_dir.exists():
    shutil.rmtree(torchair_source_dir)
torchair_source_dir.mkdir(exist_ok=True)
print("TorchAirSources:", torchair_source_dir.resolve())


#### 3.3.2 Ascend C kernel 实现

TorchAir 样例中的 `AddCustom` 仍然使用 Ascend C kernel 完成 `z = x + y`。


In [ ]:
%%writefile TorchAirSources/add_custom_kernel.asc
#include <cstdint>

#include "kernel_operator.h"

namespace ascendc_ops {
constexpr uint32_t BLOCK_LENGTH = 2048U;

// 每个 block 处理 2048 个 float 元素，计算 z = x + y。
__global__ __vector__ void add_custom(__gm__ float *x, __gm__ float *y, __gm__ float *z)
{
    AscendC::InitSocState();

    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    xGm.SetGlobalBuffer(x + block_idx * BLOCK_LENGTH, BLOCK_LENGTH);
    yGm.SetGlobalBuffer(y + block_idx * BLOCK_LENGTH, BLOCK_LENGTH);
    zGm.SetGlobalBuffer(z + block_idx * BLOCK_LENGTH, BLOCK_LENGTH);

    AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    AscendC::LocalTensor<float> xLocal = ubAllocator.Alloc<float>(BLOCK_LENGTH);
    AscendC::LocalTensor<float> yLocal = ubAllocator.Alloc<float>(BLOCK_LENGTH);
    AscendC::LocalTensor<float> zLocal = ubAllocator.Alloc<float>(BLOCK_LENGTH);

    AscendC::DataCopy(xLocal, xGm, BLOCK_LENGTH);
    AscendC::DataCopy(yLocal, yGm, BLOCK_LENGTH);
    AscendC::PipeBarrier<PIPE_ALL>();

    AscendC::Add(zLocal, xLocal, yLocal, BLOCK_LENGTH);
    AscendC::PipeBarrier<PIPE_ALL>();

    AscendC::DataCopy(zGm, zLocal, BLOCK_LENGTH);
    AscendC::PipeBarrier<PIPE_ALL>();
}
}  // namespace ascendc_ops


#### 3.3.3 PyTorch schema 和 Meta 注册

TorchAir 前端入图需要先让 PyTorch 认识 `torch.ops.ascendc_ops.ascendc_add`。schema 定义 Python 可调用的算子接口；Meta 实现提供 `torch.compile` 捕获阶段需要的输出 shape / dtype 信息。


In [ ]:
%%writefile TorchAirSources/add_custom_pytorch.cpp
#include <torch/extension.h>

namespace {
constexpr int64_t ROW_NUM = 8;
constexpr int64_t COL_NUM = 2048;

// Meta 实现只做形状和类型推导，不执行真实 NPU 计算。
at::Tensor AscendcAddMeta(const at::Tensor &x, const at::Tensor &y)
{
    TORCH_CHECK(x.sizes() == y.sizes(), "x and y must have the same shape.");
    TORCH_CHECK(x.dim() == 2 && x.size(0) == ROW_NUM && x.size(1) == COL_NUM,
        "ascendc_add only supports shape [8, 2048] in this sample.");
    TORCH_CHECK(x.scalar_type() == at::ScalarType::Float && y.scalar_type() == at::ScalarType::Float,
        "ascendc_add only supports float tensors in this sample.");
    return at::empty_like(x);
}
}  // namespace

// 注册 PyTorch 自定义算子 schema，使 Python 侧可通过 torch.ops.ascendc_ops.ascendc_add 调用。
TORCH_LIBRARY_FRAGMENT(ascendc_ops, m)
{
    m.def("ascendc_add(Tensor x, Tensor y) -> Tensor");
}

// 注册 Meta 实现，使 torch.compile 捕获图时能够推导输出 Tensor 描述。
TORCH_LIBRARY_IMPL(ascendc_ops, Meta, m)
{
    m.impl("ascendc_add", TORCH_FN(AscendcAddMeta));
}


#### 3.3.4 GE 原型、推导和执行入口

这部分与上一节的 GE 侧流程一致：`REG_OP(AddCustom)` 声明算子原型，`InferShape()` / `InferDataType()` 提供推导逻辑，`Execute()` 在 GE 运行到节点时启动 Ascend C kernel。


In [ ]:
%%writefile TorchAirSources/add_custom_ge.asc
#include <cstdint>
#include <iostream>

#include "graph/custom_op.h"
#include "graph/operator_reg.h"
#include "kernel_operator.h"

namespace ascendc_ops {
constexpr uint32_t BLOCK_NUM = 8U;
constexpr uint32_t BLOCK_LENGTH = 2048U;

// add_custom 核函数实现在 add_custom_kernel.asc 中，这里声明后由 GE 执行入口启动。
__global__ __vector__ void add_custom(__gm__ float *x, __gm__ float *y, __gm__ float *z);
}  // namespace ascendc_ops

namespace {
constexpr size_t INPUT_X_INDEX = 0U;
constexpr size_t INPUT_Y_INDEX = 1U;
constexpr size_t OUTPUT_Z_INDEX = 0U;

bool IsSupportedShape(const gert::Tensor &tensor)
{
    const gert::Shape &shape = tensor.GetStorageShape();
    return shape.GetDimNum() == 2U && shape.GetDim(0) == ascendc_ops::BLOCK_NUM &&
        shape.GetDim(1) == ascendc_ops::BLOCK_LENGTH;
}
}  // namespace

namespace ge {
class AddCustom : public EagerExecuteOp, public ShapeInferOp {
public:
    graphStatus Execute(gert::EagerOpExecutionContext *ctx) override
    {
        if (ctx == nullptr) {
            std::cerr << "Execute context is null." << std::endl;
            return GRAPH_FAILED;
        }

        const gert::Tensor *x = ctx->GetInputTensor(INPUT_X_INDEX);
        const gert::Tensor *y = ctx->GetInputTensor(INPUT_Y_INDEX);
        if ((x == nullptr) || (y == nullptr)) {
            std::cerr << "Input tensor is null." << std::endl;
            return GRAPH_FAILED;
        }
        if ((x->GetDataType() != DT_FLOAT) || (y->GetDataType() != DT_FLOAT)) {
            std::cerr << "AddCustom only supports float in this sample." << std::endl;
            return GRAPH_FAILED;
        }
        if (!IsSupportedShape(*x) || !IsSupportedShape(*y)) {
            std::cerr << "AddCustom expects both inputs to have shape [8, 2048]." << std::endl;
            return GRAPH_FAILED;
        }

        gert::Tensor *z = ctx->MallocOutputTensor(OUTPUT_Z_INDEX, x->GetShape(), x->GetFormat(), x->GetDataType());
        if (z == nullptr) {
            std::cerr << "Failed to malloc output tensor." << std::endl;
            return GRAPH_FAILED;
        }

        ascendc_ops::add_custom<<<ascendc_ops::BLOCK_NUM, nullptr, ctx->GetStream()>>>(
            static_cast<float *>(const_cast<void *>(x->GetAddr())),
            static_cast<float *>(const_cast<void *>(y->GetAddr())),
            static_cast<float *>(z->GetAddr()));
        return GRAPH_SUCCESS;
    }

    graphStatus InferShape(gert::InferShapeContext *ctx) override
    {
        if (ctx == nullptr) {
            return GRAPH_FAILED;
        }
        const gert::Shape *xShape = ctx->GetInputShape(INPUT_X_INDEX);
        gert::Shape *zShape = ctx->GetOutputShape(OUTPUT_Z_INDEX);
        if ((xShape == nullptr) || (zShape == nullptr)) {
            return GRAPH_FAILED;
        }
        *zShape = *xShape;
        return GRAPH_SUCCESS;
    }

    graphStatus InferDataType(gert::InferDataTypeContext *ctx) override
    {
        if (ctx == nullptr) {
            return GRAPH_FAILED;
        }
        return ctx->SetOutputDataType(OUTPUT_Z_INDEX, ctx->GetInputDataType(INPUT_X_INDEX));
    }
};

REG_OP(AddCustom)
    .INPUT(x, TensorType({DT_FLOAT}))
    .INPUT(y, TensorType({DT_FLOAT}))
    .OUTPUT(z, TensorType({DT_FLOAT}))
    .OP_END_FACTORY_REG(AddCustom);

REG_AUTO_MAPPING_OP(AddCustom);
}  // namespace ge


#### 3.3.5 TorchAir converter 和测试脚本

动态库加载后，PyTorch 侧的 `ascendc_ops::ascendc_add` 生效。converter 把 `torch.compile` 捕获到的 FX 节点转换成 GE `AddCustom` 节点，后续执行仍进入 GE 侧 `Execute()`。


In [ ]:
%%writefile TorchAirSources/add_custom_test.py
import os
from typing import Any

import torch
import torch_npu
import torchair
from torchair.ge import Tensor


LIB_PATH = os.environ.get("ASCENDC_OPS_LIB", os.path.join(os.path.dirname(__file__), "build/libascendc_ops.so"))
# 加载编译生成的动态库，使PyTorch侧注册的ascendc_ops::ascendc_add接口生效
torch.ops.load_library(LIB_PATH)


# 注册TorchAir converter：torch.compile捕获到ascendc_add节点后，由该函数转换成GE AddCustom节点
@torchair.register_fx_node_ge_converter(torch.ops.ascendc_ops.ascendc_add.default)
def convert_ascendc_add(x: Tensor, y: Tensor, z: Tensor = None, meta_outputs: Any = None):
    return torchair.ge.custom_op(
        "AddCustom",
        inputs={
            "x": x,
            "y": y,
        },
        outputs=["z"],
    )


class AddCustomModel(torch.nn.Module):
    def forward(self, x, y):
        # 模型中直接调用PyTorch侧注册的接口，后续由torch.compile捕获该调用
        return torch.ops.ascendc_ops.ascendc_add(x, y)


def main():
    if torch.npu.device_count() == 0:
        print("跳过 TorchAir 图模式运行验证：当前环境未检测到可用 NPU 设备。")
        return

    shape = [8, 2048]
    torch.manual_seed(0)
    # 在CPU侧构造输入和标准结果，便于最后和NPU图模式执行结果做数值对比
    x_cpu = torch.rand(shape, device="cpu", dtype=torch.float32)
    y_cpu = torch.rand(shape, device="cpu", dtype=torch.float32)

    model = AddCustomModel().npu()
    config = torchair.CompilerConfig()
    npu_backend = torchair.get_npu_backend(compiler_config=config)
    # 使用TorchAir后端编译模型，fullgraph=True确保forward中的ascendc_add被整图捕获
    opt_model = torch.compile(model, fullgraph=True, backend=npu_backend, dynamic=False)

    # 在NPU上执行编译后的图，执行链路会经过TorchAir转换出的GE AddCustom节点
    output = opt_model(x_cpu.npu(), y_cpu.npu()).cpu()
    golden = torch.add(x_cpu, y_cpu)
    # 校验图模式输出与CPU标准加法结果一致，证明本样例的接入链路和计算结果符合预期
    torch.testing.assert_close(output, golden, rtol=1e-4, atol=1e-4)
    print("TorchAir GE AddCustom sample success.")


if __name__ == "__main__":
    main()


#### 3.3.6 编译并运行 TorchAir 扩展样例

该样例需要当前 Python 环境已安装匹配版本的 `torch`、`torch_npu` 和 `torchair`。下面的代码会先检查依赖；缺少 TorchAir 环境时会跳过，不影响上一节 GE C++ 主样例。样例默认使用 `dav-3510`，Atlas A2/A3 环境可改为 `dav-2201`。


In [ ]:
%%writefile TorchAirSources/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU architecture: dav-3510, dav-2201")

find_package(ASC REQUIRED)
project(ge_torchair LANGUAGES ASC CXX)
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
find_package(Python3 COMPONENTS Interpreter Development REQUIRED)

execute_process(
    COMMAND ${CMAKE_COMMAND} -E env TORCH_DEVICE_BACKEND_AUTOLOAD=0
            ${Python3_EXECUTABLE} -c "import torch; print(torch.utils.cmake_prefix_path)"
    OUTPUT_STRIP_TRAILING_WHITESPACE
    OUTPUT_VARIABLE TORCH_CMAKE_PREFIX_PATH
)
find_package(Torch REQUIRED HINTS ${TORCH_CMAKE_PREFIX_PATH})

add_library(ascendc_ops SHARED
    add_custom_kernel.asc
    add_custom_pytorch.cpp
    add_custom_ge.asc
)
set_target_properties(ascendc_ops PROPERTIES LINKER_LANGUAGE ASC)
separate_arguments(TORCH_CXX_FLAGS_LIST UNIX_COMMAND "${TORCH_CXX_FLAGS}")
target_compile_options(ascendc_ops PRIVATE
    $<$<COMPILE_LANGUAGE:CXX>:${TORCH_CXX_FLAGS_LIST}>
)
target_include_directories(ascendc_ops PRIVATE
    ${CMAKE_SOURCE_DIR}
    ${TORCH_INCLUDE_DIRS}
    ${ASCEND_HOME_PATH}/include
    ${ASCEND_HOME_PATH}/include/graph
    ${ASCEND_HOME_PATH}/include/register
    ${ASCEND_HOME_PATH}/include/external
)
target_link_directories(ascendc_ops PRIVATE ${ASCEND_HOME_PATH}/lib64)
target_link_libraries(ascendc_ops PRIVATE
    -Wl,--no-as-needed
    Python3::Python
    gert
    ${TORCH_LIBRARIES}
    -Wl,--as-needed
)


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

cann_home = Path(os.environ.get("ASCEND_HOME_PATH", "/usr/local/Ascend/cann"))
set_env = cann_home / "set_env.sh"
if not set_env.exists():
    raise FileNotFoundError(
        f"未找到 CANN 环境初始化脚本: {set_env}。请确认 CANN 安装路径，或先设置 ASCEND_HOME_PATH。"
    )

commands = [
    "set -e",
    f"source {shlex.quote(str(set_env))}",
    "cmake -S TorchAirSources -B TorchAirSources/build -DCMAKE_ASC_ARCHITECTURES=dav-3510",
    "cmake --build TorchAirSources/build -j",
    "cd TorchAirSources",
    "python3 add_custom_test.py",
]
subprocess.run(["bash", "-lc", " && ".join(commands)], check=True)


---
### 3.4 本章小结

本章通过两个样例展示自定义算子进入 GE 图的两种入口：

- GE C++ 直接构图样例：Host 侧直接创建 `ge::Graph`，把 CANN 内置 `Add` 和自定义 `AddCustom` 放入同一张图；自定义算子侧需要提供 `REG_OP(AddCustom)` 原型、shape/dtype 推导、`Execute()` 执行入口和 Ascend C kernel，并通过 `Session::RunGraph` 在线编译运行。
- PyTorch + TorchAir 前端入图样例：模型侧调用 `torch.ops.ascendc_ops.ascendc_add`，`torch.compile` 捕获 PyTorch 图，TorchAir converter 将该节点转换为 GE `AddCustom`；GE 侧仍需要算子原型、推导、`Execute()` 和 Ascend C kernel，差异主要在前端入口和节点生成方式。


---
## 4. 课后练习

1. （判断题）GE 在线执行场景中，程序可以在运行时构造 `ge::Graph`，并通过 `Session::RunGraph` 传入输入 Tensor、获取输出 Tensor。

2. （判断题）离线编译执行链路会生成 OM。部署程序通常通过 ACL 加载 OM，并使用模型执行接口运行。

3. （判断题）CANN 内置算子和自定义算子都可以作为 GE 图中的节点参与图编译和图执行。

4. （判断题）自定义算子只要实现 Ascend C kernel，就可以被 GE 识别并完成入图执行，不需要声明算子原型和注册映射。

5. （单选题）C++ Host 侧直接构造包含 CANN 内置 `Add` 和自定义 `AddCustom` 的 GE 图时，样例采用的执行方式是：  
A. 使用 `Session::RunGraph` 在线执行  
B. 使用 `atc` 生成 OM 后执行  
C. 只调用单算子 ACL 接口执行  
D. 只在 Python 中模拟计算

6. （单选题）判断一个自定义算子能否进入 GE 图时，首先需要确认的是：  
A. 是否安装了 matplotlib  
B. GE 是否能够识别该算子的原型、输入输出和推导逻辑  
C. 是否删除了所有 Host 侧代码  
D. 是否一定生成了 OM 文件

7. （单选题）`REG_OP(AddCustom)` 在样例中的作用是：  
A. 声明 GE 可识别的 `AddCustom` 算子原型  
B. 启动 Ascend C kernel  
C. 执行 `Session::RunGraph`  
D. 将输出 Tensor 拷贝回 Host

8. （单选题）样例中 CANN 内置 `Add` 和自定义 `AddCustom` 的主要差异体现在：  
A. CANN 内置 `Add` 不能作为 GE 图节点  
B. CANN 内置 `Add` 的原型、推导和实现由 CANN 内置算子包提供，`AddCustom` 需要用户补齐  
C. `AddCustom` 只能在 Python 中运行  
D. CANN 内置 `Add` 必须通过 `Execute()` 动态调用

9. （单选题）`Execute()` 中使用 `ctx->GetStream()` 的目的是：  
A. 使用 GE 当前执行流启动 Ascend C kernel，使自定义算子计算纳入 GE 调度  
B. 读取 CMake 构建目录  
C. 设置 `ASCEND_HOME_PATH`  
D. 生成自定义算子原型头文件

10. （单选题）如果希望运行 C++ Host 直接构图的 `Add` + `AddCustom` 混合入图样例，运行前需要重点确认的是：  
A. `ASCEND_CUSTOM_OPP_PATH` 包含自定义算子输出目录  
B. `plt.plot()` 能正常绘图  
C. 每次运行都重新生成 OM 文件  
D. 删除 `Sources/build` 目录

11. （多选题）关于在线执行和离线编译执行，下列说法正确的是：  
A. 在线执行可以在进程内构造 `ge::Graph` 并调用 `Session::RunGraph` 运行  
B. 在线执行链路不生成 `.om` 文件  
C. 离线编译链路会先生成 OM，再由部署程序加载执行  
D. 离线编译执行 OM 时仍然通过 `Session::RunGraph` 运行

12. （多选题）自定义 `AddCustom` 能够进入并运行在 GE 图中，需要提供哪些内容？  
A. 算子原型声明  
B. shape / dtype 推导逻辑  
C. 执行入口和 Ascend C kernel  
D. 将 GE 节点关联到用户实现的注册映射



**执行以下代码获取答案。**


In [ ]:
!cat ./answer/06.03_answers.txt